# Notebook 07b — Probe submissions (same model, different post-processing)
Solar Filament Segmentation Challenge 2026 — Kaggle edition

**Why.** The first submission (Baseline-01 epoch 50 + BN re-estimation + flip TTA + min 400 px + closing 10) scored **0.28** on the public leaderboard,
while our validation estimate was **0.372**. To find out *why*, we submit three variants of the SAME model, each isolating one ingredient:

| file | model / probability | post-processing | validation estimate (official protocol or primary record) |
|---|---|---|---|
| `P1_baseline_raw.csv` | Baseline-01 best checkpoint as saved, no TTA | threshold 0.5, no size filter, no closing | 0.290 |
| `P2_e50_bn_tta_nofilter.csv` | epoch-50, BN re-estimated, flip TTA | threshold 0.5, no size filter, no closing | ~0.304 (Notebook 06, primary record) |
| `P3_e50_bn_tta_min200.csv` | epoch-50, BN re-estimated, flip TTA | threshold 0.5, min 200 px, no closing | ~0.351 (Notebook 06, primary record) |
| *(already submitted)* recipe | same | min 400 px + closing 10 | 0.372 -> **public LB 0.28** |

Reading the leaderboard results: if P1 scores far below 0.28 (about 0.20), the whole scale is shifted and the recipe gains DO transfer;
if P1 is about 0.27-0.29, the post-processing gains do not transfer to the test set. The leaderboard shows only 2 decimals, so differences under 0.01 cannot be resolved.

**Setup:** Add Input -> the competition AND the finished Baseline-01 FULL output. GPU on. Save & Run All (about 10-15 minutes). Download the three CSVs from the Output tab.
Use at most one submission per file (5 per day limit).

In [1]:
# ============================================================
# CONFIGURATION — the only cell you should need to edit
# ============================================================

# Competition data: attach it via "Add Input". Path is auto-detected (notebooks 1 and 2 used the first one).
KAGGLE_DATA_CANDIDATES = [
    "/kaggle/input/competitions/filament-segmentation-2026",
    "/kaggle/input/filament-segmentation-2026",
]

RUN_MODE = "FULL"               # <<< REAL Baseline-01.  ("SMOKE" | "PILOT" | "FULL", see the markdown cell above)
assert RUN_MODE in ("SMOKE", "PILOT", "FULL"), "RUN_MODE must be SMOKE, PILOT or FULL"
DEBUG_MODE = (RUN_MODE == "SMOKE")   # reduced-image-count shortcuts exist ONLY in SMOKE mode

OUTPUT_DIR = "/kaggle/working"  # kept only if you "Save Version"; see markdown above

SEED = 42
RESUME = True                   # resume from latest valid checkpoint if one exists

# ---- experiment parameters (Baseline-01) — same field names as Notebook 03 ----
CFG = {
    "experiment_id": "Baseline-01" if RUN_MODE == "FULL" else f"Baseline-01-{RUN_MODE}",
    "train_resolution": 512 if DEBUG_MODE else 1024,
    "batch_size": 2 if DEBUG_MODE else 4,
    "epochs": {"SMOKE": 20, "PILOT": 3, "FULL": 50}[RUN_MODE],
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "model": "UNet-base32",
    "loss": "BCE+Dice",
    "prob_threshold": 0.5,       # single configurable baseline threshold (NOT tuned)
    "min_instance_px": 0,        # no size filtering in the baseline
    "debug_n_train": 24,         # smoke-test only
    "debug_n_val": 8,            # smoke-test only
    # ---- run-infrastructure settings (do not change the methodology) ----
    "use_amp": True,             # automatic mixed precision for training / val loss
    "num_workers": None,         # None -> min(4, cpu_count)
    "pq_eval_every": 10,         # monitor val PQ every N epochs (0 = only final). Monitoring only.
    "snapshot_every": 10,        # keep a full-state snapshot every N epochs
    "ckpt_every": 2,             # write latest_checkpoint.pth every N epochs (and always at the last epoch)
    "fsync_checkpoints": False,  # fsync was a suspected cause of slow saves on Kaggle disk; atomic rename is kept
}

print(f"RUN_MODE={RUN_MODE} (DEBUG_MODE={DEBUG_MODE}) | resolution={CFG['train_resolution']} "
      f"| epochs={CFG['epochs']} | batch={CFG['batch_size']} | AMP={CFG['use_amp']}")

RUN_MODE=FULL (DEBUG_MODE=False) | resolution=1024 | epochs=50 | batch=4 | AMP=True


In [2]:
import os, sys, json, random, re, warnings, math, shutil, time, hashlib, platform, gc, datetime, copy
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available (no CPU fallback). Kaggle panel -> Session options -> Accelerator -> GPU.")
DEVICE = torch.device("cuda:0")
CFG["num_workers"] = min(4, os.cpu_count() or 2)
print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} | workers {CFG['num_workers']}")

OUT = Path(OUTPUT_DIR) / "probes"; OUT.mkdir(parents=True, exist_ok=True)

# ---- locate the saved Baseline-01 FULL output (attached as an Input) ----
# Searches for the CHECKPOINT FILE anywhere under /kaggle/input (the notebook's title does not matter).
hits = sorted(Path("/kaggle/input").rglob("baseline_best.pth"))
print("baseline_best.pth found at:", [str(h) for h in hits] if hits else "NOWHERE")
if not hits:
    print("\nWhat is mounted under /kaggle/input (depth <= 3):")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 3:
            pth = [f for f in files if f.endswith(".pth")]
            print(f"  {root}  | files: {len(files)}" + (f" | .pth: {pth[:6]}" if pth else ""))
    raise FileNotFoundError(
        "No baseline_best.pth under /kaggle/input.\n"
        "The input you added contains no checkpoint files. Most likely the saved version has an EMPTY Output tab "
        "(Quick Save, still running, or failed). Open that notebook on Kaggle -> the version -> 'Output' tab: it must list "
        "baseline_best.pth, latest_checkpoint.pth, checkpoint_epoch_0X0.pth ... If it is empty, re-run the Baseline-01 FULL "
        "notebook with 'Save & Run All' and wait for the version to finish, then add THAT version's output as the input.")
PREV = max((h.parent for h in hits), key=lambda p: p.stat().st_mtime)
print("Baseline-01 output folder used:", PREV)
print("files:", sorted(p.name for p in PREV.iterdir()))

torch 2.11.0+cu128 | Tesla T4 | workers 4
baseline_best.pth found at: ['/kaggle/input/notebooks/parsabehjattabrizi/baseline-best/full/baseline_best.pth']
Baseline-01 output folder used: /kaggle/input/notebooks/parsabehjattabrizi/baseline-best/full
files: ['baseline_best.pth', 'baseline_best_meta.json', 'baseline_report.txt', 'baseline_results.csv', 'checkpoint_epoch_010.pth', 'checkpoint_epoch_020.pth', 'checkpoint_epoch_030.pth', 'checkpoint_epoch_040.pth', 'checkpoint_epoch_050.pth', 'experiment_log.csv', 'iou_dice_dist.png', 'latest_checkpoint.pth', 'latest_checkpoint_prev.pth', 'per_image_metrics.csv', 'run_config.json', 'split_metadata.csv', 'training_curve.png', 'training_history.csv', 'validation_gallery.png', 'validation_gallery_worst3.png']


In [3]:
# ============================================================
# DATA DISCOVERY on Kaggle (/kaggle/input) — same discovery logic as Notebook 03
# ============================================================

def acquire_data_root():
    for c in KAGGLE_DATA_CANDIDATES:
        if Path(c).exists():
            return Path(c)
    base = Path("/kaggle/input")
    if base.exists() and any(base.rglob("train_images")):
        print("WARNING: standard path not found; using /kaggle/input and auto-discovery.")
        return base
    raise FileNotFoundError(
        "Competition data not found under /kaggle/input.\n"
        "Fix: right-hand panel -> Add Input -> search 'filament-segmentation-2026' -> Add.")

DATA_ROOT = acquire_data_root()
print("DATA_ROOT:", DATA_ROOT)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def discover_dataset(root):
    """Find train/test image dirs and the COCO annotation JSON. Fail loudly."""
    img_dirs = []
    for d in sorted({p.parent for p in root.rglob("*") if p.suffix.lower() in IMG_EXTS}):
        n = sum(1 for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
        img_dirs.append((d, n))
        print(f"  image dir: {d.relative_to(root)}  ({n} images)")
    train_cands = [d for d, _ in img_dirs
                   if "train" in str(d).lower() and "test" not in str(d).lower()]
    test_cands = [d for d, _ in img_dirs
                  if "test" in str(d).lower() and "train" not in str(d).lower()]
    if len(train_cands) != 1 or len(test_cands) != 1:
        raise RuntimeError(
            f"Ambiguous dataset layout. train candidates={train_cands}, "
            f"test candidates={test_cands}. Inspect the listing above.")
    coco = None
    for p in root.rglob("*.json"):
        try:
            with open(p) as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d:
                coco = p; break
        except Exception:
            pass
    if coco is None:
        raise RuntimeError(f"No COCO-style annotation JSON found under {root}")
    return train_cands[0], test_cands[0], coco

TRAIN_DIR, TEST_DIR, ANN_PATH = discover_dataset(DATA_ROOT)

train_paths = sorted(p for p in TRAIN_DIR.iterdir() if p.suffix.lower() in IMG_EXTS)
test_paths  = sorted(p for p in TEST_DIR.iterdir()  if p.suffix.lower() in IMG_EXTS)
name_to_path = {p.name: p for p in train_paths}

print(f"\ntrain images: {len(train_paths)} | test images: {len(test_paths)}")
print(f"annotations: {ANN_PATH.name}")
assert len(train_paths) > 0 and len(test_paths) > 0

DATA_ROOT: /kaggle/input/competitions/filament-segmentation-2026
  image dir: MAGFiLO_1.0_Kaggle_2026/test/test_images  (180 images)
  image dir: MAGFiLO_1.0_Kaggle_2026/train/train_images  (707 images)

train images: 707 | test images: 180
annotations: MAGFiLO_1.0_Annotations_kaggle2026_train.json


In [4]:
# ============================================================
# ANNOTATIONS + BASELINE GT REPRESENTATION  (unchanged from Notebook 03)
# Baseline target = FIRST annotation record per file (sorted by id).
# ============================================================

def load_annotations(path):
    with open(path) as f:
        coco = json.load(f)
    images_info = {im["id"]: im for im in coco["images"]}
    anns_by_image = defaultdict(list)
    for a in coco["annotations"]:
        anns_by_image[a["image_id"]].append(a)
    return coco, images_info, anns_by_image

COCO, images_info, anns_by_image = load_annotations(ANN_PATH)
print(f"JSON records: {len(images_info)} | annotations: {len(COCO['annotations'])}")

file_to_records = defaultdict(list)
for img_id, info in images_info.items():
    fname = Path(info["file_name"]).name
    if fname in name_to_path:
        file_to_records[fname].append(img_id)

missing = len(images_info) - sum(len(v) for v in file_to_records.values())
if missing:
    raise RuntimeError(f"{missing} JSON records reference files not on disk")

file_to_primary = {f: sorted(ids)[0] for f, ids in file_to_records.items()}
primary_ids = sorted(file_to_primary.values())
print(f"physical files: {len(file_to_primary)} | primary records: {len(primary_ids)}")

def ann_to_mask(ann, h, w):
    m = np.zeros((h, w), np.uint8)
    for poly in ann["segmentation"]:
        pts = np.asarray(poly, np.float64).reshape(-1, 2)
        cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
    return m.astype(bool)

def build_gt_masks(img_id):
    """Full-resolution instance masks for one record."""
    info = images_info[img_id]
    masks = [ann_to_mask(a, info["height"], info["width"])
             for a in anns_by_image[img_id]]
    masks = [m for m in masks if m.any()]
    if not masks:
        raise ValueError(f"Record {img_id} has no non-empty GT masks")
    return masks

def load_image(path):
    img = np.array(Image.open(path))
    if img.ndim == 3:
        img = img[..., :3].mean(axis=2)
    return img.astype(np.float32)

JSON records: 1154 | annotations: 8199
physical files: 707 | primary records: 707


In [5]:
from pycocotools import mask as mask_utils

# ============================================================
# RLE (pycocotools, Fortran order, 2048x2048)  — kept for the later submission notebook
# ============================================================

def encode_rle(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle

def decode_rle(rle):
    r = dict(rle)
    if isinstance(r["counts"], str):
        r["counts"] = r["counts"].encode("utf-8")
    return mask_utils.decode(r).astype(bool)

_t = np.zeros((2048, 2048), bool)
_t[100:130, 400:900] = True; _t[1500:1510, 30:80] = True
_rt = decode_rle(encode_rle(_t))
assert _rt.shape == (2048, 2048) and (_rt == _t).all(), "RLE round-trip mismatch"
assert decode_rle(encode_rle(np.zeros((2048, 2048), bool))).sum() == 0
print("RLE round-trip checks passed (2048x2048, incl. empty mask)")

RLE round-trip checks passed (2048x2048, incl. empty mask)


In [6]:
# ============================================================
# SPLIT — date-grouped 80/20 (Notebook 02 A3). Same function, same seed as Notebook 03.
# ============================================================

FN_RE = re.compile(r"^(\d{4})(\d{2})(\d{2})\d{6}([A-Za-z])h\.", re.I)

def create_split(primary_ids, val_frac=0.2, seed=SEED):
    rows = []
    for img_id in primary_ids:
        fname = Path(images_info[img_id]["file_name"]).name
        m = FN_RE.match(fname)
        if not m:
            raise ValueError(f"Unparseable filename: {fname}")
        rows.append({"img_id": img_id, "file": fname,
                     "date": f"{m[1]}-{m[2]}-{m[3]}", "station": m[4].upper()})
    df = pd.DataFrame(rows)
    dates = sorted(df["date"].unique())
    rng = random.Random(seed); rng.shuffle(dates)
    n_val_dates = int(round(len(dates) * val_frac))
    val_dates = set(dates[:n_val_dates])
    df["split"] = df["date"].map(lambda d: "val" if d in val_dates else "train")
    return df

split_df = create_split(primary_ids)
train_ids = split_df.loc[split_df.split == "train", "img_id"].tolist()
val_ids   = split_df.loc[split_df.split == "val", "img_id"].tolist()

if DEBUG_MODE:                       # smoke-test shortcut — must be OFF for Baseline-01
    train_ids = train_ids[:CFG["debug_n_train"]]
    val_ids   = val_ids[:CFG["debug_n_val"]]

SPLIT_HASH = hashlib.sha1(",".join(sorted(
    split_df.loc[split_df.split == "val", "file"])).encode()).hexdigest()[:12]

split_df.to_csv(OUT / "split_metadata.csv", index=False)
print(f"train: {len(train_ids)} | val: {len(val_ids)} (date-grouped) | split hash: {SPLIT_HASH}")
assert not set(split_df[split_df.split == "train"].date) & \
           set(split_df[split_df.split == "val"].date)

train: 568 | val: 139 (date-grouped) | split hash: 0cc2d729a5b1


In [7]:
# ============================================================
# DATASET — fixed-size tensors. Preprocessing and augmentation (h/v flips only)
# are IDENTICAL to Notebook 03.
# ============================================================

class FilamentDataset(Dataset):
    def __init__(self, record_ids, resolution, augment=False):
        self.ids = record_ids
        self.res = resolution
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = images_info[img_id]
        fname = Path(info["file_name"]).name
        img = load_image(name_to_path[fname])
        h, w = img.shape

        gt = np.zeros((h, w), bool)
        for a in anns_by_image[img_id]:
            gt |= ann_to_mask(a, h, w)

        img = cv2.resize(img, (self.res, self.res), interpolation=cv2.INTER_AREA)
        gt = cv2.resize(gt.astype(np.uint8), (self.res, self.res),
                        interpolation=cv2.INTER_NEAREST).astype(bool)

        lo, hi = np.percentile(img, [1, 99])
        img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)

        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy(); gt = np.fliplr(gt).copy()
            if random.random() < 0.5:
                img = np.flipud(img).copy(); gt = np.flipud(gt).copy()

        return {"image": torch.from_numpy(img)[None],
                "mask": torch.from_numpy(gt.astype(np.float32))[None],
                "img_id": img_id}

def _worker_init(_):
    cv2.setNumThreads(0)

train_ds = FilamentDataset(train_ids, CFG["train_resolution"], augment=True)
val_ds   = FilamentDataset(val_ids, CFG["train_resolution"], augment=False)

# generator is re-seeded every epoch (SEED + epoch) so shuffling/augmentation is reproducible
# and identical whether or not the run was resumed.
train_gen = torch.Generator(); train_gen.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True,
                          generator=train_gen, worker_init_fn=_worker_init)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True,
                        worker_init_fn=_worker_init)

# sample check WITHOUT creating DataLoader iterators (avoids the dangling-worker noise seen in Notebook 03)
_s = train_ds[0]
print("sample check:", tuple(_s["image"].shape), tuple(_s["mask"].shape),
      "| train batches/epoch:", len(train_loader), "| val images:", len(val_ds))
del _s

sample check: (1, 1024, 1024) (1, 1024, 1024) | train batches/epoch: 142 | val images: 139


In [8]:
# ============================================================
# MODEL — plain U-Net (base 32), UNCHANGED.  Loss — BCE + soft Dice, UNCHANGED
# (logits are cast to float32 inside the loss so AMP cannot affect the loss math).
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, cin=1, base=32):
        super().__init__()
        self.e1, self.e2 = ConvBlock(cin, base), ConvBlock(base, base*2)
        self.e3, self.e4 = ConvBlock(base*2, base*4), ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*8, base*16)
        self.u4 = nn.ConvTranspose2d(base*16, base*8, 2, 2); self.d4 = ConvBlock(base*16, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2);  self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2);  self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2);    self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        m = self.mid(self.pool(e4))
        d = self.d4(torch.cat([self.u4(m), e4], 1))
        d = self.d3(torch.cat([self.u3(d), e3], 1))
        d = self.d2(torch.cat([self.u2(d), e2], 1))
        d = self.d1(torch.cat([self.u1(d), e1], 1))
        return self.out(d)

def build_model():
    return UNet(1, 32).to(DEVICE)

def build_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

def bce_dice_loss(logits, target, eps=1.0):
    logits = logits.float()
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    dice = (2 * inter + eps) / (prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps)
    return bce + (1 - dice).mean()

USE_AMP = bool(CFG["use_amp"]) and DEVICE.type == "cuda"

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)

def build_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)

EXPECTED_PARAMS = 7_765_409      # parameter count reported by the Notebook 03 smoke test
_tmp_model = build_model()
N_PARAMS = sum(p.numel() for p in _tmp_model.parameters())
del _tmp_model
print(f"parameters: {N_PARAMS:,} | AMP: {USE_AMP}")

parameters: 7,765,409 | AMP: True


In [9]:
# ============================================================
# CONSISTENCY WITH BASELINE-01 + HELPERS
# ============================================================
prev_split = pd.read_csv(PREV / "split_metadata.csv")
assert set(prev_split.loc[prev_split.split == "val", "file"]) == set(split_df.loc[split_df.split == "val", "file"]), \
    "Validation split differs from Baseline-01 — stop."
print("Validation split identical to Baseline-01:", len(val_ids), "files | hash", SPLIT_HASH)

hist_path, meta_path = PREV / "training_history.csv", PREV / "baseline_best_meta.json"
if hist_path.exists():
    n_ep = len(pd.read_csv(hist_path)); print(f"training_history.csv: {n_ep} epochs")
    if n_ep < CFG["epochs"]:
        raise RuntimeError(f"Baseline-01 input is INCOMPLETE ({n_ep}/{CFG['epochs']} epochs). Attach the finished version.")
if meta_path.exists():
    meta = json.loads(meta_path.read_text()); print(f"baseline_best.pth = epoch {meta['best_epoch']} (val loss {meta['best_val_loss']:.4f})")
prev_res = pd.read_csv(PREV / "baseline_results.csv").iloc[0] if (PREV / "baseline_results.csv").exists() else None
if prev_res is not None:
    print(f"Baseline-01 saved result: PQ pooled {prev_res['PQ_pooled']:.4f} | mean/img {prev_res['PQ_mean_per_image']:.4f}")

def summarize(items):
    tp = sum(r["TP"] for r in items); fp = sum(r["FP"] for r in items); fn = sum(r["FN"] for r in items)
    ious = [x for r in items for x in r["tp_ious"]]
    denom = tp + 0.5 * fp + 0.5 * fn
    return {"PQ_pooled": (sum(ious) / denom) if denom else 0.0,
            "PQ_mean_per_image": float(np.mean([r["PQ"] for r in items])) if items else 0.0,
            "TP": tp, "FP": fp, "FN": fn, "mean_tp_iou": float(np.mean(ious)) if ious else 0.0}

vsplit = split_df[split_df.img_id.isin(val_ids)]
date_half = {d: "AB"[i % 2] for i, d in enumerate(sorted(vsplit.date.unique()))}
img_half = {r.img_id: date_half[r.date] for r in vsplit.itertuples()}

def load_model(path):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m = build_model(); m.load_state_dict(sd); m.eval()
    return m

def recalibrate_bn(model, n_batches=60):
    """Re-estimate BatchNorm running statistics on ~240 training images (weights untouched)."""
    m = copy.deepcopy(model)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.reset_running_stats(); mod.momentum = None
    g = torch.Generator(); g.manual_seed(SEED)
    loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                        drop_last=True, generator=g, worker_init_fn=_worker_init)
    it = iter(loader); m.train()
    with torch.no_grad():
        for _ in range(n_batches):
            b = next(it)
            with autocast_ctx():
                m(b["image"].to(DEVICE))
    del it; m.eval()
    return m

Validation split identical to Baseline-01: 139 files | hash 0cc2d729a5b1
training_history.csv: 50 epochs
baseline_best.pth = epoch 42 (val loss 0.3619)
Baseline-01 saved result: PQ pooled 0.2881 | mean/img 0.2974


In [10]:
# ============================================================
# HELPERS (post-processing, flip-TTA prediction, upsampling)
# ============================================================
def postprocess(prob, thr, min_px, close_r):
    binary = (prob > thr).astype(np.uint8)
    if close_r:
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                                  cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_r + 1, 2 * close_r + 1)))
    n, labels = cv2.connectedComponents(binary, connectivity=8)
    areas = np.bincount(labels.ravel(), minlength=n)
    return [labels == k for k in range(1, n) if areas[k] > min_px]

@torch.no_grad()
def predict_views(model, img_path, resolution, tta=True):
    """Returns (plain prob, flip-TTA-averaged prob) at model resolution + original image shape."""
    img = load_image(img_path)
    small = cv2.resize(img, (resolution, resolution), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(small, [1, 99])
    small = np.clip((small - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)
    x = torch.from_numpy(small)[None, None].to(DEVICE)
    views = [None, [3], [2], [2, 3]] if tta else [None]
    outs = []
    for dims in views:
        xi = x if dims is None else torch.flip(x, dims)
        p = torch.sigmoid(model(xi))
        if dims is not None:
            p = torch.flip(p, dims)
        outs.append(p[0, 0].cpu().numpy())
    return outs[0], np.mean(outs, axis=0), img.shape

def upsample(p, shape):
    h, w = shape
    return cv2.resize(p, (w, h), interpolation=cv2.INTER_LINEAR)

In [11]:
# ============================================================
# MODELS + PROBE SUBMISSIONS
# ============================================================
m_best = load_model(PREV / "baseline_best.pth")                              # exactly as saved
m_e50 = recalibrate_bn(load_model(PREV / "checkpoint_epoch_050.pth"))        # BN statistics re-estimated

PROBES = {
    "P1_baseline_raw":          ("best", dict(tta=False, thr=0.5, min_px=0,   close=0)),
    "P2_e50_bn_tta_nofilter":   ("e50",  dict(tta=True,  thr=0.5, min_px=0,   close=0)),
    "P3_e50_bn_tta_min200":     ("e50",  dict(tta=True,  thr=0.5, min_px=200, close=0)),
}
rows = {k: [] for k in PROBES}
counts = {k: [] for k in PROBES}
bad_shape = 0
for path in tqdm(test_paths, desc="test inference"):
    plain_b, _, shp = predict_views(m_best, path, CFG["train_resolution"], tta=False)
    plain_e, tta_e, _ = predict_views(m_e50, path, CFG["train_resolution"], tta=True)
    if tuple(shp) != (2048, 2048):
        bad_shape += 1
    for name, (which, p) in PROBES.items():
        small = plain_b if which == "best" else (tta_e if p["tta"] else plain_e)
        preds = postprocess(upsample(small, shp), p["thr"], p["min_px"], p["close"])
        counts[name].append(len(preds))
        for k, m_ in enumerate(preds):
            rows[name].append({"filament_id": f"{path.stem}_{k}", "segmentation_rle": encode_rle(m_)["counts"]})

print("images not 2048x2048:", bad_shape)
summary = []
for name in PROBES:
    df = pd.DataFrame(rows[name], columns=["filament_id", "segmentation_rle"])
    assert df.filament_id.is_unique
    df.to_csv(Path(OUTPUT_DIR) / f"{name}.csv", index=False)
    c = np.array(counts[name])
    summary.append({"file": f"{name}.csv", "filaments": len(df), "images_with_pred": int((c > 0).sum()),
                    "mean_per_image": round(float(c.mean()), 1), "median_per_image": float(np.median(c))})
summ = pd.DataFrame(summary); print(summ.to_string(index=False))
summ.to_csv(OUT / "probe_summary.csv", index=False)
print("\nFiles written to", OUTPUT_DIR, ":", sorted(p.name for p in Path(OUTPUT_DIR).glob("P*.csv")))
print("Submit each CSV once, then send me the three public scores.")

test inference:   0%|          | 0/180 [00:00<?, ?it/s]

images not 2048x2048: 0
                      file  filaments  images_with_pred  mean_per_image  median_per_image
       P1_baseline_raw.csv       2225               179            12.4              12.0
P2_e50_bn_tta_nofilter.csv       2155               178            12.0              11.5
  P3_e50_bn_tta_min200.csv       1654               177             9.2               9.0

Files written to /kaggle/working : ['P1_baseline_raw.csv', 'P2_e50_bn_tta_nofilter.csv', 'P3_e50_bn_tta_min200.csv']
Submit each CSV once, then send me the three public scores.
